In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('fpb_test.csv')

# Display the first 5 rows of the DataFrame
display(df.head())

,text,label,source,clean_text
0,the share capital of alma media corporation bu...,Neutral,FinancialPhraseBank,the share capital of alma media corporation bu...
1,the eu commission said earlier it had fined th...,Fear,FinancialPhraseBank,the eu commission said earlier it had fined th...
2,"kesko pursues a strategy of healthy , focused ...",Optimism,FinancialPhraseBank,kesko pursues a strategy of healthy focused gr...
3,down to eur5 .9 m h1 '09 3 august 2009 - finni...,Sadness,FinancialPhraseBank,down to eurNUM NUM m hNUM NUM NUM august NUM f...
4,"cencorp would focus on the development , manuf...",Neutral,FinancialPhraseBank,cencorp would focus on the development manufac...


### Log in to Hugging Face

To access models from Hugging Face, especially if they are private or if you need to bypass rate limits, you should log in using your Hugging Face token. Store your token securely in Colab's secrets manager, naming it `HF_TOKEN`.

In [2]:
from huggingface_hub import login
from google.colab import userdata

# Retrieve the Hugging Face token from Colab's secrets
hf_token = userdata.get('HF_TOKEN')

# Log in to Hugging Face
if hf_token:
    login(token=hf_token)
    print("Successfully logged in to Hugging Face.")
else:
    print("Hugging Face token not found in Colab secrets. Please add it as 'HF_TOKEN'.")

Successfully logged in to Hugging Face.


### load Gemma model and tokenizer

In [3]:
# install transformers
!pip install transformers accelerate -q

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# model id
model_name = "google/gemma-3-4b-it"

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

### data prep, few-shot

create a prompt that includes a few examples of the task (the 'shots') and then the text we want the model to classify

split the dataset into a small set for examples and the rest for classification

In [4]:
# unique labels are...
print("Unique labels in the dataset:", df['label'].unique())
unique_labels = df['label'].unique().tolist()

df_neutral = df[df['label'] == 'Neutral'].copy()
df_fear = df[df['label'] == 'Fear'].copy()
df_optimism = df[df['label'] == 'Optimism'].copy()
df_sadness = df[df['label'] == 'Sadness'].copy()
df_joy = df[df['label'] == 'Joy'].copy()

neutral_example = df_neutral.sample(n=1, random_state=30) # reproducibility
fear_example = df_fear.sample(n=1, random_state=30)
optimism_example = df_optimism.sample(n=1, random_state=30)
sadness_example = df_sadness.sample(n=1, random_state=30)
joy_example = df_joy.sample(n=1, random_state=30)

few_shot_examples = pd.concat([neutral_example, fear_example, optimism_example, sadness_example, joy_example])

# # Shuffle the dataframe to ensure random selection of few-shot examples
# df_shuffled = df.sample(frac=1, random_state=30).reset_index(drop=True)

# # Select a few examples for few-shot prompting
# num_few_shot_examples = 5 # 10 would be too much and take too long
# few_shot_examples = df_shuffled.head(num_few_shot_examples)
# inference_data = df_shuffled.tail(len(df_shuffled) - num_few_shot_examples).reset_index(drop=True)

inference_data = df.copy()

for r in few_shot_examples.itertuples(index=False):
    inference_data = inference_data[inference_data.text != r.text]

Unique labels in the dataset: ['Neutral' 'Fear' 'Optimism' 'Sadness' 'Joy']


### few-shot inference

iterate through inference_data

we use the Gemma chat prompt here due to the extended few-shot prompt being more suitable for it, based on examples in https://deepwiki.com/google-gemini/gemma-cookbook/4.3-prompting-techniques.

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# classify function
def classify(text, candidate_labels):

    few_shot_prompt_prefix = f"Your task is to classify text into one of the following categories: {', '.join(unique_labels)}. The first few samples have been pre-filled for you:\n\n"

    for index, row in few_shot_examples.iterrows():
        few_shot_prompt_prefix += f"Example Text: '{row['text']}'\nExample Label: {row['label']}\n\n"

    # Construct the prompt in the Gemma chat format
    chat_prompt = [
        {"role": "user", "content": few_shot_prompt_prefix + f"Now, based on these examples, classify the following text into one of the provided categories. Respond with *only* the category label, nothing else.\n\nText: {text}\n"}
    ]

    prompt = tokenizer.apply_chat_template(chat_prompt, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=50, do_sample=False)

    # Decode only the newly generated tokens, skipping the input prompt
    generated_token_ids = output[0][inputs.input_ids.shape[1]:]
    generated_text = tokenizer.decode(generated_token_ids, skip_special_tokens=True).strip()

    predicted_label = "Unknown" # default value

    # remove any "category" labels that may be left in the generated text
    cleaned_generated_text = generated_text.replace("Category:", "").strip()

    # case-insensitive matching
    generated_text_lower = cleaned_generated_text.lower()

    # Attempt to find the best match by checking if the label is *in* the generated text
    for label in candidate_labels:
        if label.lower() in generated_text_lower:
            predicted_label = label
            break

    return predicted_label

y_true = inference_data['label'].tolist() # all the correct labels
y_pred = [classify(text, unique_labels) for text in inference_data['text']] # all the predicted labels

# print(y_true)
# print(y_pred)


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [6]:
# basic metrics
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average=None, labels=unique_labels, zero_division=0)

# Create a DataFrame for per-class F1 scores
per_class_f1_df = pd.DataFrame({
    'Category': unique_labels,
    'Precision': precision,
    'Recall': recall,
    'F1 Score': f1,
    'Support': support
})

print("Per-class F1 Scores:")
display(per_class_f1_df)

# For overall metrics, we can still calculate them (e.g., weighted average)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', labels=unique_labels, zero_division=0)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', labels=unique_labels, zero_division=0)

# export metrics (overall weighted metrics)
results_dict = {
    'Metric': ['Accuracy', 'Precision (Weighted)', 'Recall (Weighted)', 'F1 Score (Weighted)', 'Precision (Macro)', 'Recall (Macro)', 'F1 Score (Macro)'] and ['Accuracy', 'Precision (Weighted)', 'Recall (Weighted)', 'F1 Score (Weighted)', 'Precision (Macro)', 'Recall (Macro)', 'F1 Score (Macro)'],
    'Value': [accuracy, precision_weighted, recall_weighted, f1_weighted, precision_macro, recall_macro, f1_macro]
}

# create dataframe
results_df = pd.DataFrame(results_dict)
display(results_df)

dashboard_dict = {
    'model': 'Gemma 3 4B',
    'strategy': 'few-shot',
    'accuracy': accuracy,
    'f1_macro': f1_macro,
    'f1_weighted': f1_weighted,
    'f1_fear': per_class_f1_df.at[1, 'F1 Score'],
    'f1_joy': per_class_f1_df.at[4, 'F1 Score'],
    'f1_neutral': per_class_f1_df.at[0, 'F1 Score'],
    'f1_optimism': per_class_f1_df.at[2, 'F1 Score'],
    'f1_sadness': per_class_f1_df.at[3, 'F1 Score']
}

dashboard_df = pd.DataFrame(dashboard_dict, index=[0])

# export to csv
dashboard_df.to_csv('few_shot_gemma_dashboard.csv', index=False)

print("Results saved to 'few_shot_gemma_dashboard.csv'")
display(dashboard_df)

Per-class F1 Scores:


,Category,Precision,Recall,F1 Score,Support
0,Neutral,0.791209,0.668213,0.724528,431
1,Fear,0.037500,1.000000,0.072289,6
2,Optimism,0.725000,0.297436,0.421818,195
3,Sadness,0.672727,0.451220,0.540146,82
4,Joy,0.063492,0.500000,0.112676,8


,Metric,Value
0,Accuracy,0.544321
1,Precision (Weighted),0.745544
2,Recall (Weighted),0.544321
3,F1 Score (Weighted),0.609631
4,Precision (Macro),0.457986
5,Recall (Macro),0.583374
6,F1 Score (Macro),0.374292


Results saved to 'few_shot_gemma_dashboard.csv'


,model,strategy,accuracy,f1_macro,f1_weighted,f1_fear,f1_joy,f1_neutral,f1_optimism,f1_sadness
0,Gemma 3 4B,few-shot,0.544321,0.374292,0.609631,0.072289,0.112676,0.724528,0.421818,0.540146
